# Issue #8: Track Weights During Training (LoRA vs DoRA)

This notebook reproduces the weight decomposition analysis from the DoRA paper (Figure 3).
We train one LoRA and one DoRA model, capturing magnitude and direction metrics at each
evaluation step, then visualize the correlation between magnitude and direction changes.

**Expected result**: DoRA shows lower correlation between magnitude and direction updates
than LoRA, indicating more independent learning of these components.

In [ ]:
# Cell 1: Setup
!pip install -q torch transformers datasets peft fire matplotlib numpy scipy

# Clone the repo (skip if running locally)
import os
if not os.path.exists('dsa5106-group-2'):
    !git clone https://github.com/jeykori/dsa5106-group-2.git

%cd dsa5106-group-2

# Download datasets
if not os.path.exists('datasets/commonsense_170k.json'):
    !bash scripts/init-datasets.sh

# Add reproduction/ to path
import sys
sys.path.insert(0, './reproduction')

print('Setup complete.')

In [ ]:
# Cell 2: Hugging Face login (needed for gated Llama model)
from huggingface_hub import login
login()

In [ ]:
# Cell 3: WeightTrackingCallback

import torch
import numpy as np
from transformers import TrainerCallback
from dora import DoraLayer


class WeightTrackingCallback(TrainerCallback):
    """Tracks magnitude and direction changes for LoRA/DoRA layers during training."""

    def __init__(self, model, adapter_type):
        self.adapter_type = adapter_type
        self.tracking_data = {"adapter": adapter_type, "steps": [], "layers": {}}
        self._snapshot_initial_weights(model)

    @torch.no_grad()
    def _snapshot_initial_weights(self, model):
        """Store initial magnitude and direction for each adapter layer."""
        self.initial_state = {}
        for name, module in model.named_modules():
            if self.adapter_type == "dora" and isinstance(module, DoraLayer):
                m_init = module.m.detach().clone()
                dir_init = module.base_layer.weight.detach().clone()
                dir_init = dir_init / torch.linalg.norm(dir_init, dim=1, keepdim=True)
                self.initial_state[name] = {"m": m_init, "direction": dir_init}
                self.tracking_data["layers"][name] = {"magnitude_change": [], "direction_change": []}

            elif self.adapter_type == "lora" and hasattr(module, 'lora_A'):
                # PEFT LoRA layer
                if not hasattr(module, 'base_layer'):
                    continue
                w0 = module.base_layer.weight.detach().clone()
                m_init = torch.linalg.norm(w0, dim=1, keepdim=True)
                dir_init = w0 / m_init
                self.initial_state[name] = {"m": m_init, "direction": dir_init}
                self.tracking_data["layers"][name] = {"magnitude_change": [], "direction_change": []}

    @torch.no_grad()
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        step = state.global_step
        self.tracking_data["steps"].append(step)

        for name, module in model.named_modules():
            if name not in self.initial_state:
                continue

            init = self.initial_state[name]

            if self.adapter_type == "dora" and isinstance(module, DoraLayer):
                # Magnitude: directly from m parameter
                m_current = module.m.detach()
                mag_change = torch.norm(m_current - init["m"]).item()

                # Direction: v_new = base_weight + (B @ A) * scaling
                delta_v = (module.lora_B.weight @ module.lora_A.weight) * module.scaling
                v_new = module.base_layer.weight.detach() + delta_v
                dir_current = v_new / torch.linalg.norm(v_new, dim=1, keepdim=True)

            elif self.adapter_type == "lora" and hasattr(module, 'lora_A'):
                # Post-hoc decompose W_eff = W0 + B @ A * scaling
                w0 = module.base_layer.weight.detach()
                lora_A = module.lora_A['default'].weight.detach()
                lora_B = module.lora_B['default'].weight.detach()
                scaling = module.scaling['default']
                w_eff = w0 + (lora_B @ lora_A) * scaling

                m_current = torch.linalg.norm(w_eff, dim=1, keepdim=True)
                mag_change = torch.norm(m_current - init["m"]).item()

                dir_current = w_eff / m_current
            else:
                continue

            # Direction change: average angle in degrees
            cos_sim = torch.nn.functional.cosine_similarity(init["direction"], dir_current, dim=1)
            cos_sim = torch.clamp(cos_sim, -1.0, 1.0)
            angle_deg = torch.acos(cos_sim).mean().item() * (180.0 / np.pi)

            self.tracking_data["layers"][name]["magnitude_change"].append(mag_change)
            self.tracking_data["layers"][name]["direction_change"].append(angle_deg)

        print(f"[WeightTracking] Step {step}: recorded {len(self.initial_state)} layers")


print('WeightTrackingCallback defined.')

In [ ]:
# Cell 4: Training wrapper

import torch
import transformers
from datasets import load_dataset
from lora import inject_lora
from dora import inject_dora, merge_and_unload_dora
from utils import generate_prompt

# Copied from reproduction/finetune.py
def tokenize_prompt(data, tokenizer):
    user_prompt = generate_prompt({**data, "output": ""})
    full_prompt = generate_prompt(data) + tokenizer.eos_token
    tokenized_user_prompt = tokenizer(user_prompt, padding=False)
    tokenized = tokenizer(full_prompt, padding=False)
    user_prompt_len = len(tokenized_user_prompt["input_ids"])
    labels = tokenized["input_ids"].copy()
    labels = [-100] * user_prompt_len + labels[user_prompt_len:]
    tokenized["labels"] = labels
    return tokenized


def train_with_tracking(
    adapter="dora",
    lora_r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    learning_rate=2e-4,
    sample_size=3000,
    val_set_size=120,
    num_epochs=3,
    micro_batch_size=4,
    batch_size=16,
    eval_steps=40,
    save_steps=40,
    model_name="unsloth/llama-3.2-3b",
    dataset_path="./datasets/commonsense_170k.json",
    output_dir=None,
    target_modules=None,
):
    if target_modules is None:
        target_modules = ["q_proj", "k_proj", "v_proj", "up_proj", "down_proj"]
    if output_dir is None:
        output_dir = f"./tracking_output/{adapter}-r{lora_r}"

    gradient_accumulation_steps = batch_size // micro_batch_size

    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", dtype=torch.bfloat16
    )

    if adapter == "lora":
        model = inject_lora(model, r=lora_r, lora_alpha=lora_alpha,
                            lora_dropout=lora_dropout, target_modules=target_modules)
    elif adapter == "dora":
        model = inject_dora(model, r=lora_r, lora_alpha=lora_alpha,
                            lora_dropout=lora_dropout, target_modules=target_modules)

    # Create callback after adapter injection
    callback = WeightTrackingCallback(model, adapter)

    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"

    data = load_dataset("json", data_files=dataset_path)
    if sample_size is not None:
        data["train"] = data["train"].shuffle(seed=42).select(range(sample_size))

    data_split = data["train"].train_test_split(test_size=val_set_size, shuffle=True, seed=42)
    data_train = data_split["train"].shuffle().map(lambda x: tokenize_prompt(x, tokenizer))
    data_val = data_split["test"].shuffle().map(lambda x: tokenize_prompt(x, tokenizer))

    trainer = transformers.Trainer(
        model=model,
        train_dataset=data_train,
        eval_dataset=data_val,
        callbacks=[callback],
        args=transformers.TrainingArguments(
            output_dir=output_dir,
            gradient_accumulation_steps=gradient_accumulation_steps,
            num_train_epochs=num_epochs,
            learning_rate=learning_rate,
            eval_strategy="steps",
            eval_steps=eval_steps,
            save_strategy="steps",
            save_steps=save_steps,
            per_device_train_batch_size=micro_batch_size,
            bf16=True,
            save_total_limit=3,
            gradient_checkpointing=True,
            optim="adamw_torch",
            warmup_steps=100,
        ),
        data_collator=transformers.DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable} | Total: {total} | %: {100 * trainable / total:.4f}")

    trainer.train()

    # Cleanup GPU memory
    del model, trainer
    torch.cuda.empty_cache()

    return callback.tracking_data


print('train_with_tracking() defined.')

In [ ]:
# Cell 5: Train DoRA model with weight tracking
dora_tracking = train_with_tracking(adapter="dora", lora_r=4, lora_alpha=8)
print(f"DoRA tracking: {len(dora_tracking['steps'])} eval steps, {len(dora_tracking['layers'])} layers")

In [ ]:
# Cell 6: Train LoRA model with weight tracking
lora_tracking = train_with_tracking(adapter="lora", lora_r=4, lora_alpha=8)
print(f"LoRA tracking: {len(lora_tracking['steps'])} eval steps, {len(lora_tracking['layers'])} layers")

In [ ]:
# Cell 7: Save tracking data
import json
os.makedirs('./tracking_output', exist_ok=True)
with open('./tracking_output/dora_tracking.json', 'w') as f:
    json.dump(dora_tracking, f, indent=2)
with open('./tracking_output/lora_tracking.json', 'w') as f:
    json.dump(lora_tracking, f, indent=2)
print('Tracking data saved.')

In [ ]:
# Cell 8: Scatter plot — Magnitude vs Direction change (DoRA paper Figure 3)

import matplotlib.pyplot as plt
from scipy import stats

def get_layer_type(name):
    for t in ['q_proj', 'k_proj', 'v_proj', 'up_proj', 'down_proj']:
        if t in name:
            return t
    return 'other'

LAYER_COLORS = {
    'q_proj': '#1f77b4', 'k_proj': '#ff7f0e', 'v_proj': '#2ca02c',
    'up_proj': '#d62728', 'down_proj': '#9467bd', 'other': '#8c564b'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, tracking, title in [
    (axes[0], dora_tracking, 'DoRA'),
    (axes[1], lora_tracking, 'LoRA'),
]:
    all_mag, all_dir = [], []
    for layer_name, metrics in tracking['layers'].items():
        layer_type = get_layer_type(layer_name)
        color = LAYER_COLORS[layer_type]
        mag = metrics['magnitude_change']
        dirn = metrics['direction_change']
        ax.scatter(dirn, mag, c=color, alpha=0.5, s=15, label=layer_type)
        all_mag.extend(mag)
        all_dir.extend(dirn)

    # Pearson correlation
    if all_mag and all_dir:
        r, p = stats.pearsonr(all_dir, all_mag)
        ax.set_title(f'{title} (r={r:.3f}, p={p:.2e})')
    else:
        ax.set_title(title)

    ax.set_xlabel('Direction Change (degrees)')
    ax.set_ylabel('Magnitude Change (L2 norm)')

    # Deduplicate legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=8)

fig.suptitle('Magnitude vs Direction Change: DoRA vs LoRA', fontsize=14)
plt.tight_layout()
plt.savefig('./tracking_output/scatter_mag_vs_dir.png', dpi=150)
plt.show()

In [ ]:
# Cell 9: Time series — Magnitude and direction change over training steps

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (tracking, adapter_name) in enumerate([
    (dora_tracking, 'DoRA'), (lora_tracking, 'LoRA')
]):
    steps = tracking['steps']
    for layer_name, metrics in tracking['layers'].items():
        layer_type = get_layer_type(layer_name)
        color = LAYER_COLORS[layer_type]
        axes[0, col].plot(steps, metrics['magnitude_change'], color=color, alpha=0.3, linewidth=0.8)
        axes[1, col].plot(steps, metrics['direction_change'], color=color, alpha=0.3, linewidth=0.8)

    axes[0, col].set_title(f'{adapter_name} — Magnitude Change')
    axes[0, col].set_xlabel('Step')
    axes[0, col].set_ylabel('||m_current - m_init||')

    axes[1, col].set_title(f'{adapter_name} — Direction Change')
    axes[1, col].set_xlabel('Step')
    axes[1, col].set_ylabel('Angle (degrees)')

plt.tight_layout()
plt.savefig('./tracking_output/timeseries_mag_dir.png', dpi=150)
plt.show()

In [ ]:
# Cell 10: Correlation bar chart — Per layer type, LoRA vs DoRA

from collections import defaultdict

def compute_correlations_by_type(tracking):
    grouped = defaultdict(lambda: {'mag': [], 'dir': []})
    for layer_name, metrics in tracking['layers'].items():
        lt = get_layer_type(layer_name)
        grouped[lt]['mag'].extend(metrics['magnitude_change'])
        grouped[lt]['dir'].extend(metrics['direction_change'])

    result = {}
    for lt, data in grouped.items():
        if len(data['mag']) > 2:
            r, _ = stats.pearsonr(data['dir'], data['mag'])
            result[lt] = r
    return result

dora_corr = compute_correlations_by_type(dora_tracking)
lora_corr = compute_correlations_by_type(lora_tracking)

layer_types = sorted(set(list(dora_corr.keys()) + list(lora_corr.keys())))
x = np.arange(len(layer_types))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, [dora_corr.get(lt, 0) for lt in layer_types], width, label='DoRA', color='#2ca02c')
ax.bar(x + width/2, [lora_corr.get(lt, 0) for lt in layer_types], width, label='LoRA', color='#1f77b4')

ax.set_ylabel('Pearson Correlation (mag vs dir)')
ax.set_title('Magnitude-Direction Correlation by Layer Type')
ax.set_xticks(x)
ax.set_xticklabels(layer_types)
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('./tracking_output/correlation_bar.png', dpi=150)
plt.show()

print('\nCorrelation summary:')
for lt in layer_types:
    print(f'  {lt}: DoRA={dora_corr.get(lt, float("nan")):.3f}, LoRA={lora_corr.get(lt, float("nan")):.3f}')